In [ ]:
""" Sequence Alignment and Analysis with Biopython
  Task 3: Pairwise Sequence Alignment using Biopython
  Implements Needleman-Wunsch (global) and Smith-Waterman (local) alignment.
  experiments with substitution matrices (BLOSUM62, BLOSUM45, PAM250) and gap penalties.
  Evaluates alignment quality via score, identity %, and similarity %."""

** Import Libraries**

In [15]:
from itertools import combinations
from Bio import SeqIO
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [18]:
# Load sequences from the fasta file
def load_sequences(filepath, file_format="fasta"):
    """Load all sequences from a FASTA file."""
    sequences = list(SeqIO.parse(filepath, file_format))
    if not sequences:
        raise ValueError(f"No sequences found in {filepath}")
    print(f"Loaded {len(sequences)} sequence(s) from '{filepath}'")
    for s in sequences:
        print(f"  > {s.id} | length={len(s.seq)}")
    return sequences  

In [19]:
seq1_records = load_sequences("hemoglobin_gallusgallus.fasta")
seq2_records = load_sequences("hemoglobin_homosapien.fasta")

seq1 = str(seq1_records[0].seq)
seq2 = str(seq2_records[0].seq)

Loaded 1 sequence(s) from 'hemoglobin_gallusgallus.fasta'
  > NP_990820.1 | length=147
Loaded 1 sequence(s) from 'hemoglobin_homosapien.fasta'
  > NP_000509.1 | length=147


In [20]:
# Compute alignment quality metrics
def compute_metrics(aligned_seq1, aligned_seq2):
    """
    Compute identity and similarity from two aligned sequences.
    Identity   = exact matches / aligned residues (no gaps)
    Similarity = (matches + conservative substitutions) / aligned residues
    """
    conservative = [
        {"S", "T", "A"}, {"N", "D", "E", "Q"}, {"R", "K"},
        {"I", "L", "V", "M"}, {"F", "Y", "W"}, {"H", "R", "K"},
        {"G", "A"}, {"P"},
    ]

    length = len(aligned_seq1)
    gaps = sum(1 for a, b in zip(aligned_seq1, aligned_seq2) if a == "-" or b == "-")
    aligned_length = length - gaps
    identical = sum(1 for a, b in zip(aligned_seq1, aligned_seq2) if a == b and a != "-")

    similar = identical
    for a, b in zip(aligned_seq1, aligned_seq2):
        if a != b and a != "-" and b != "-":
            for group in conservative:
                if a.upper() in group and b.upper() in group:
                    similar += 1
                    break

    identity   = (identical / aligned_length * 100) if aligned_length > 0 else 0
    similarity = (similar  / aligned_length * 100) if aligned_length > 0 else 0

    return {
        "alignment_length": length,
        "aligned_residues": aligned_length,
        "gaps": gaps,
        "identical": identical,
        "similar": similar,
        "identity_pct": round(identity, 2),
        "similarity_pct": round(similarity, 2),
    }

In [21]:
# Global alignment: Needleman-Wunsch Algorithm
def global_alignment(seq1, seq2, matrix_name="BLOSUM62", gap_open=-10, gap_extend=-0.5):
    """Needleman-Wunsch global alignment."""
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.substitution_matrix = substitution_matrices.load(matrix_name)
    aligner.open_gap_score   = gap_open
    aligner.extend_gap_score = gap_extend

    best = next(iter(aligner.align(seq1, seq2)))
    metrics = compute_metrics(str(best[0]), str(best[1]))

    print(f"── Global Alignment (Needleman-Wunsch) ──")
    print(f"Matrix: {matrix_name} | Gap open: {gap_open} | Gap extend: {gap_extend}")
    print(f"Score       : {best.score:.2f}")
    print(f"Identity    : {metrics['identity_pct']}%  ({metrics['identical']}/{metrics['aligned_residues']})")
    print(f"Similarity  : {metrics['similarity_pct']}%")
    print(f"Gaps        : {metrics['gaps']}")
    print(f"\nSeq1: {str(best[0])[:80]}")
    print(f"Seq2: {str(best[1])[:80]}")
    return best, metrics

global_best, global_metrics = global_alignment(seq1, seq2)

── Global Alignment (Needleman-Wunsch) ──
Matrix: BLOSUM62 | Gap open: -10 | Gap extend: -0.5
Score       : 554.00
Identity    : 69.39%  (102/147)
Similarity  : 84.35%
Gaps        : 0

Seq1: MVHWTAEEKQLITGLWGKVNVAECGAEALARLLIVYPWTQRFFASFGNLSSPTAILGNPMVRAHGKKVLTSFGDAVKNLD
Seq2: MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLD


In [22]:
# Local alignment: Smith-Waterman
def local_alignment(seq1, seq2, matrix_name="BLOSUM62", gap_open=-10, gap_extend=-0.5):
    """Smith-Waterman local alignment."""
    aligner = PairwiseAligner()
    aligner.mode = "local"
    aligner.substitution_matrix = substitution_matrices.load(matrix_name)
    aligner.open_gap_score   = gap_open
    aligner.extend_gap_score = gap_extend

    best = next(iter(aligner.align(seq1, seq2)))
    metrics = compute_metrics(str(best[0]), str(best[1]))

    print(f"── Local Alignment (Smith-Waterman) ──")
    print(f"Matrix: {matrix_name} | Gap open: {gap_open} | Gap extend: {gap_extend}")
    print(f"Score       : {best.score:.2f}")
    print(f"Identity    : {metrics['identity_pct']}%  ({metrics['identical']}/{metrics['aligned_residues']})")
    print(f"Similarity  : {metrics['similarity_pct']}%")
    print(f"Gaps        : {metrics['gaps']}")
    print(f"\nSeq1: {str(best[0])[:80]}")
    print(f"Seq2: {str(best[1])[:80]}")
    return best, metrics

local_best, local_metrics = local_alignment(seq1, seq2)


── Local Alignment (Smith-Waterman) ──
Matrix: BLOSUM62 | Gap open: -10 | Gap extend: -0.5
Score       : 554.00
Identity    : 69.39%  (102/147)
Similarity  : 84.35%
Gaps        : 0

Seq1: MVHWTAEEKQLITGLWGKVNVAECGAEALARLLIVYPWTQRFFASFGNLSSPTAILGNPMVRAHGKKVLTSFGDAVKNLD
Seq2: MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLD


In [23]:
# Matrix & gap penalty comparison
import pandas as pd   # for a clean table display

def compare_matrices(seq1, seq2, mode="global"):
    """Compare alignment scores across matrices and gap penalty configs."""
    matrices    = ["BLOSUM62", "BLOSUM45", "PAM250"]
    gap_configs = [(-10, -0.5, "Standard"), (-6, -0.5, "Low open"), (-10, -1.0, "High extend")]
    rows = []

    for matrix_name in matrices:
        for gap_open, gap_extend, label in gap_configs:
            aligner = PairwiseAligner()
            aligner.mode = mode
            aligner.substitution_matrix = substitution_matrices.load(matrix_name)
            aligner.open_gap_score   = gap_open
            aligner.extend_gap_score = gap_extend

            best    = next(iter(aligner.align(seq1, seq2)))
            metrics = compute_metrics(str(best[0]), str(best[1]))
            rows.append({
                "Matrix": matrix_name, "Gap Config": label,
                "Score": round(best.score, 2),
                "Identity %": metrics["identity_pct"],
                "Similarity %": metrics["similarity_pct"],
            })

    df = pd.DataFrame(rows)
    print(f"\nMode: {mode.upper()}")
    print(df.to_string(index=False))
    return df

global_comparison = compare_matrices(seq1, seq2, mode="global")
local_comparison  = compare_matrices(seq1, seq2, mode="local")


Mode: GLOBAL
  Matrix  Gap Config  Score  Identity %  Similarity %
BLOSUM62    Standard  554.0       69.39         84.35
BLOSUM62    Low open  554.0       69.39         84.35
BLOSUM62 High extend  554.0       69.39         84.35
BLOSUM45    Standard  668.0       69.39         84.35
BLOSUM45    Low open  668.0       69.39         84.35
BLOSUM45 High extend  668.0       69.39         84.35
  PAM250    Standard  570.0       69.39         84.35
  PAM250    Low open  570.0       69.39         84.35
  PAM250 High extend  570.0       69.39         84.35

Mode: LOCAL
  Matrix  Gap Config  Score  Identity %  Similarity %
BLOSUM62    Standard  554.0       69.39         84.35
BLOSUM62    Low open  554.0       69.39         84.35
BLOSUM62 High extend  554.0       69.39         84.35
BLOSUM45    Standard  668.0       69.39         84.35
BLOSUM45    Low open  668.0       69.39         84.35
BLOSUM45 High extend  668.0       69.39         84.35
  PAM250    Standard  570.0       69.39         84.35
 

In [26]:
# Cell 8 — All pairwise combinations
if 'seq1_records' not in globals() or 'seq2_records' not in globals():
    print("Please run Cell 2 first to load your sequences.")
else:
    seq_records = seq1_records + seq2_records

    if len(seq_records) < 2:
        print("Need at least 2 sequences to compare.")
    else:
        aligner = PairwiseAligner()
        aligner.mode = "global"
        aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
        aligner.open_gap_score   = -10
        aligner.extend_gap_score = -0.5

        rows = []
        for r1, r2 in combinations(seq_records, 2):
            s1, s2 = str(r1.seq), str(r2.seq)
            best = next(iter(aligner.align(s1, s2)))
            m = compute_metrics(str(best[0]), str(best[1]))
            rows.append({
                "Seq1":         r1.id,
                "Seq2":         r2.id,
                "Score":        round(best.score, 2),
                "Identity %":   m["identity_pct"],
                "Similarity %": m["similarity_pct"],
            })

        print(pd.DataFrame(rows).to_string(index=False))

       Seq1        Seq2  Score  Identity %  Similarity %
NP_990820.1 NP_000509.1  554.0       69.39         84.35
